In [0]:
%pip install apache-sedona

In [0]:
# Restart Python to pick up newly installed packages
dbutils.library.restartPython()

In [0]:
from sedona.spark import SedonaContext
from pyspark.sql.functions import expr, col, coalesce, lit, monotonically_increasing_id, when

In [0]:
# Initialize Sedona spatial context
sedona = SedonaContext.create(spark)

In [0]:
# Load taxi zone polygons in SRID 2263 (NY State Plane)
zones = spark.sql("""
SELECT
  zone_id,
  st_setsrid(ST_GeomFromWKT(geometry_wkt), 2263) AS zone_geom
FROM group3_gp.gold.dim_zone
WHERE geometry_wkt IS NOT NULL
""")

In [0]:
# Load raw yellow taxi data
df_init = spark.read.table("group3_gp.bronze.yellow")

In [0]:
%sql
SELECT year, COUNT(*) FROM bronze.yellow GROUP BY year ORDER BY year

In [0]:
# Consolidate duplicate columns across years using COALESCE
df_clean = (
    df_init
    .withColumn(
        "pickup_datetime_clean",
        coalesce(col("tpep_pickup_datetime"), col("pickup_datetime"))
    )
    .withColumn(
        "dropoff_datetime_clean",
        coalesce(col("tpep_dropoff_datetime"), col("dropoff_datetime"))
    )
    .withColumn(
        "vendor_id_clean",
        coalesce(col("vendor_id"), col("vendorid"))
    )
    .withColumn(
        "rate_code_clean",
        coalesce(col("rate_code"), col("ratecodeid"))
    )
    .withColumn(
        "passenger_count_clean",
        coalesce(col("passenger_count"), lit(0.0))
    )
)

In [0]:
# Drop original overlapping columns now merged above
cols_to_drop = [
    "tpep_pickup_datetime", "pickup_datetime",
    "tpep_dropoff_datetime", "dropoff_datetime",
    "vendor_id", "vendorid",
    "improvement_surcharge", "imp_surcharge",
    "store_and_fwd_flag",
    "ratecodeid", "rate_code",
    "fare_amount_invalid_flag", "passenger_count"
]

df_clean = df_clean.drop(*cols_to_drop)

In [0]:
# Rename cleaned columns to final names
df = df_clean \
    .withColumnRenamed("pickup_datetime_clean", "pickup_datetime") \
    .withColumnRenamed("dropoff_datetime_clean", "dropoff_datetime") \
    .withColumnRenamed("vendor_id_clean", "vendor_id") \
    .withColumnRenamed("rate_code_clean", "ratecode_id") \
    .withColumnRenamed("passenger_count_clean", "passenger_count")

In [0]:
# Cast columns to proper types; payment_type, vendor_id, ratecode_id stay as strings for legacy mapping in the next cell
column_types = {
    "pickup_datetime": "timestamp",
    "dropoff_datetime": "timestamp",
    "improvement_surcharge": "decimal(10,2)",
    "passenger_count": "double",
    "pulocationid": "integer",
    "dolocationid": "integer",
    "trip_distance": "double",
    "pickup_latitude": "double",
    "pickup_longitude": "double",
    "dropoff_latitude": "double",
    "dropoff_longitude": "double",
    "fare_amount": "decimal(10,2)",
    "extra": "decimal(10,2)",
    "mta_tax": "decimal(10,2)",
    "tip_amount": "decimal(10,2)",
    "tolls_amount": "decimal(10,2)",
    "total_amount": "decimal(10,2)",
    "congestion_surcharge": "decimal(10,2)",
    "airport_fee": "decimal(10,2)"
}

for col, dtype in column_types.items():
    if col in df.columns:
        df = df.withColumn(col, expr(f"try_cast({col} as {dtype})"))

df = df.withColumn("taxi_type", lit("Yellow")).withColumn("service_type", lit("Yellow"))
print(f"Initial row count: {df.count():,}")

In [0]:
# Map legacy string codes to descriptions, then cast raw columns to integer
from pyspark.sql import functions as F
df = df.withColumn(
    "payment_types",
    F.when(F.col("payment_type").isin("1", "CRD"), "Credit card")
     .when(F.col("payment_type").isin("2", "CSH"), "Cash")
     .when(F.col("payment_type").isin("3", "NOC"), "No charge")
     .when(F.col("payment_type").isin("4", "DIS"), "Dispute")
     .when(F.col("payment_type").isin("5", "UNK"), "Unknown")
     .when(F.col("payment_type").isin("6", "VOID"), "Voided trip")
     .when(F.col("payment_type") == "0", "Flex fare")
     .otherwise("Unrecognised")
)

df = df.withColumn(
    "rate_codes",
    F.when(F.col("ratecode_id").isin("1", "1.0"), "Standard rate")
     .when(F.col("ratecode_id").isin("2", "2.0"), "JFK")
     .when(F.col("ratecode_id").isin("3", "3.0"), "Newark")
     .when(F.col("ratecode_id").isin("4", "4.0"), "Nassau or Westchester")
     .when(F.col("ratecode_id").isin("5", "5.0"), "Negotiated fare")
     .when(F.col("ratecode_id").isin("6", "6.0"), "Group ride")
     .when(F.col("ratecode_id").isin("99", "99.0"), "Unknown")
     .otherwise("Unrecognised")
)

df = df.withColumn(
    "vendors",
    F.when(F.col("vendor_id").isin("1", "CMT"), "Creative Mobile Technologies")
     .when(F.col("vendor_id") == "2", "Curb Mobility")
     .when(F.col("vendor_id") == "VTS", "VeriFone")
     .when(F.col("vendor_id") == "6", "Myle Technologies")
     .when(F.col("vendor_id") == "7", "Helix")
     .otherwise("Unrecognised")
)

for c in ["payment_type", "vendor_id", "ratecode_id"]:
    df = df.withColumn(c, expr(f"try_cast({c} as integer)"))

In [0]:
# Filter out invalid rows: missing timestamps, bad fares, missing locations
df = df.filter(
    F.col("pickup_datetime").isNotNull() &
    F.col("dropoff_datetime").isNotNull() &
    (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
    (F.col("fare_amount").cast("double") >= 0)
)
print(f"Rows after filtering: {df.count()}")

In [0]:
# Add unique row IDs for spatial join tracking
base_df = df.withColumn("row_id", monotonically_increasing_id())

In [0]:
# Build point geometries for rows missing pickup/dropoff location IDs
trips_missing_pu = base_df.filter(
        F.col("pulocationid").isNull() &
        F.col("pickup_longitude").isNotNull() &
        F.col("pickup_latitude").isNotNull() &
        (F.col("pickup_longitude").cast("double") != 0) &
        (F.col("pickup_latitude").cast("double") != 0)
    ).withColumn(
        "pickup_geom",
        F.expr("""
            ST_Transform(
                ST_SetSRID(
                    ST_Point(CAST(pickup_longitude AS DOUBLE), CAST(pickup_latitude AS DOUBLE)),
                    4326
                ),
                2263
            )
        """)
    )

trips_missing_do = base_df.filter(
        F.col("dolocationid").isNull() &
        F.col("dropoff_longitude").isNotNull() &
        F.col("dropoff_latitude").isNotNull() &
        (F.col("dropoff_longitude").cast("double") != 0) &
        (F.col("dropoff_latitude").cast("double") != 0)
    ).withColumn(
        "dropoff_geom",
        F.expr("""
            ST_Transform(
                ST_SetSRID(
                    ST_Point(CAST(dropoff_longitude AS DOUBLE), CAST(dropoff_latitude AS DOUBLE)),
                    4326
                ),
                2263
            )
        """)
    )

In [0]:
# Spatial join: match lat/lon points to taxi zone polygons
pu_match = (
    trips_missing_pu.alias("t")
    .join(
        zones.alias("z"),
        F.expr("ST_Contains(z.zone_geom, t.pickup_geom)"),
        "left"
    )
    .select("t.*", F.col("z.zone_id").alias("imputed_pu_zone_id"))
)
do_match = (
    trips_missing_do.alias("t")
    .join(
        zones.alias("z"),
        F.expr("ST_Contains(z.zone_geom, t.dropoff_geom)"),
        "left"
    )
    .select("t.*", F.col("z.zone_id").alias("imputed_do_zone_id"))
)
pu_filled = pu_match.withColumn(
    "pulocationid",
    F.coalesce(F.col("pulocationid"), F.col("imputed_pu_zone_id"))
)
do_filled = do_match.withColumn(
    "dolocationid",
    F.coalesce(F.col("dolocationid"), F.col("imputed_do_zone_id"))
)

In [0]:
# Merge imputed zone IDs back, drop helper columns, filter out remaining nulls
pu_ids = pu_filled.select("row_id", "imputed_pu_zone_id")
do_ids = do_filled.select("row_id", "imputed_do_zone_id")

final_df_temp = (
    base_df
    .join(pu_ids, on="row_id", how="left")
    .join(do_ids, on="row_id", how="left")
    .withColumn("pulocationid", F.coalesce(F.col("pulocationid"), F.col("imputed_pu_zone_id")))
    .withColumn("dolocationid", F.coalesce(F.col("dolocationid"), F.col("imputed_do_zone_id")))
)
final_df = (
    final_df_temp
    .drop("dropoff_latitude","dropoff_location","dropoff_longitude","pickup_latitude","pickup_location","pickup_longitude", "payment_type", "imputed_pu_zone_id", "imputed_do_zone_id", "pickup_geom", "dropoff_geom", "vendor_id", "ratecode_id", "airport_fee", "congestion_surcharge", "passenger_count")
    .filter(
        F.col("pulocationid").isNotNull() & 
        F.col("dolocationid").isNotNull()
        )
)

In [0]:
print(final_df.count())

In [0]:
# Create silver table from cleaned and enriched data
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.silver.yellow_taxi_trips")